In [3]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [4]:
joined = gpd.read_parquet("../../data/processed/site_features/property_with_zoning_sample_all_types.parquet")
print(len(joined))
print(joined.columns)

109666
Index(['RID', 'gurasid', 'propertytype', 'valnetpropertystatus',
       'valnetpropertytype', 'dissolveparcelcount', 'valnetlotcount', 'propid',
       'superlot', 'address', 'housenumber', 'urbanity', 'Shape__Area',
       'Shape__Length', 'lot_size_proxy_sqm', 'SYM_CODE', 'LAY_CLASS',
       'EPI_NAME', 'LGA_NAME', 'EPI_TYPE', 'geometry'],
      dtype='object')


In [12]:
# only use propertytype == 1 in first version
joined_main = joined[joined["propertytype"] == 1].copy()
print(len(joined_main))

109515


In [13]:
joined_main["RID"].duplicated().mean()

np.float64(0.5438980961512122)

In [14]:
def collapse_zoning_group(group: pd.DataFrame) -> pd.Series:
    sym_nonnull = group["SYM_CODE"].dropna()
    class_nonnull = group["LAY_CLASS"].dropna()

    if len(sym_nonnull) == 0:
        primary_sym = pd.NA
    else:
        primary_sym = sym_nonnull.value_counts().index[0]

    if len(class_nonnull) == 0:
        primary_class = pd.NA
    else:
        primary_class = class_nonnull.value_counts().index[0]

    zoning_codes = sorted(sym_nonnull.astype(str).unique().tolist()) if len(sym_nonnull) else []
    zoning_classes = sorted(class_nonnull.astype(str).unique().tolist()) if len(class_nonnull) else []

    return pd.Series(
        {
            "primary_zoning_code": primary_sym,
            "primary_zoning_class": primary_class,
            "zoning_hit_count": int(group["SYM_CODE"].notna().sum()),
            "zoning_code_count": int(sym_nonnull.nunique()),
            "mixed_zoning_flag": int(sym_nonnull.nunique() > 1),
            "zoning_codes": "|".join(zoning_codes),
            "zoning_classes": "|".join(zoning_classes),
        }
    )

In [15]:
property_base = (
    joined_main.sort_values("RID")
    .groupby("RID", as_index=False)
    .first()[
        [
            "RID",
            "gurasid",
            "propertytype",
            "valnetpropertystatus",
            "valnetpropertytype",
            "dissolveparcelcount",
            "valnetlotcount",
            "propid",
            "superlot",
            "address",
            "housenumber",
            "urbanity",
            "Shape__Area",
            "Shape__Length",
            "geometry",
        ]
    ]
)

print(len(property_base))
property_base.head()

49950


,RID,gurasid,propertytype,valnetpropertystatus,valnetpropertytype,dissolveparcelcount,valnetlotcount,propid,superlot,address,housenumber,urbanity,Shape__Area,Shape__Length,geometry
0,260,49494103,1,2.0,2.0,1,1,1341,N,44 NORTHCOTE STREET ABERDARE,44,U,1439.819123,168.171421,"POLYGON ((151.36767 -32.83893, 151.36776 -32.8..."
1,319,49494159,1,2.0,2.0,1,1,3169,N,10 MEREWETHER CLOSE NORTH ROTHBURY,10,S,25509.139491,833.661825,"POLYGON ((151.32883 -32.68241, 151.32596 -32.6..."
2,327,49494166,1,2.0,2.0,1,1,3682,N,112 MATHIESON STREET BELLBIRD HEIGHTS,112,U,1129.358164,146.067118,"POLYGON ((151.33272 -32.84584, 151.33241 -32.8..."
3,338,49494176,1,2.0,2.0,1,1,3958,N,584 WOLLOMBI ROAD BELLBIRD,584,U,987.808843,132.056667,"POLYGON ((151.31499 -32.86605, 151.31464 -32.8..."
4,359,49494197,1,2.0,2.0,1,1,4739,N,41 RUSSELL STREET BRANXTON,41,U,1347.510500,165.423675,"POLYGON ((151.34805 -32.66108, 151.34786 -32.6..."


In [16]:
# groupby collapse for zoning
zoning_features = (
    joined_main.groupby("RID", as_index=False)
    .apply(collapse_zoning_group)
    .reset_index(drop=True)
)

print(len(zoning_features))
zoning_features.head()

49950


/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_38115/3010753102.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_zoning_group)


,RID,primary_zoning_code,primary_zoning_class,zoning_hit_count,zoning_code_count,mixed_zoning_flag,zoning_codes,zoning_classes
0,260,R2,Low Density Residential,1,1,0,R2,Low Density Residential
1,319,R5,Large Lot Residential,1,1,0,R5,Large Lot Residential
2,327,R2,Low Density Residential,1,1,0,R2,Low Density Residential
3,338,SP2,Infrastructure,2,2,1,R2|SP2,Infrastructure|Low Density Residential
4,359,R3,Medium Density Residential,1,1,0,R3,Medium Density Residential


In [17]:
# merge to site-level table
site_features = property_base.merge(zoning_features, on="RID", how="left")
print(len(site_features))
site_features.head()

49950


,RID,gurasid,propertytype,valnetpropertystatus,valnetpropertytype,dissolveparcelcount,valnetlotcount,propid,superlot,address,...,Shape__Area,Shape__Length,geometry,primary_zoning_code,primary_zoning_class,zoning_hit_count,zoning_code_count,mixed_zoning_flag,zoning_codes,zoning_classes
0,260,49494103,1,2.0,2.0,1,1,1341,N,44 NORTHCOTE STREET ABERDARE,...,1439.819123,168.171421,"POLYGON ((151.36767 -32.83893, 151.36776 -32.8...",R2,Low Density Residential,1,1,0,R2,Low Density Residential
1,319,49494159,1,2.0,2.0,1,1,3169,N,10 MEREWETHER CLOSE NORTH ROTHBURY,...,25509.139491,833.661825,"POLYGON ((151.32883 -32.68241, 151.32596 -32.6...",R5,Large Lot Residential,1,1,0,R5,Large Lot Residential
2,327,49494166,1,2.0,2.0,1,1,3682,N,112 MATHIESON STREET BELLBIRD HEIGHTS,...,1129.358164,146.067118,"POLYGON ((151.33272 -32.84584, 151.33241 -32.8...",R2,Low Density Residential,1,1,0,R2,Low Density Residential
3,338,49494176,1,2.0,2.0,1,1,3958,N,584 WOLLOMBI ROAD BELLBIRD,...,987.808843,132.056667,"POLYGON ((151.31499 -32.86605, 151.31464 -32.8...",SP2,Infrastructure,2,2,1,R2|SP2,Infrastructure|Low Density Residential
4,359,49494197,1,2.0,2.0,1,1,4739,N,41 RUSSELL STREET BRANXTON,...,1347.510500,165.423675,"POLYGON ((151.34805 -32.66108, 151.34786 -32.6...",R3,Medium Density Residential,1,1,0,R3,Medium Density Residential


In [18]:
site_features["lot_size_proxy_sqm"] = site_features["Shape__Area"]
site_features["has_zoning"] = site_features["primary_zoning_code"].notna().astype(int)

# Distribution of primary zoning
site_features["primary_zoning_code"].value_counts(dropna=False).head(20)

primary_zoning_code
R2     18373
R1      6505
R3      4709
R4      2643
MU1     2196
RU1     2114
SP2     1970
RE1     1651
E1      1255
RU5     1049
C4       970
E4       869
R5       745
RU2      697
E2       637
C2       575
C3       452
RU4      334
SP5      318
E3       313
Name: count, dtype: int64

In [19]:
# Proportion of mixed zoning
site_features["mixed_zoning_flag"].value_counts(normalize=True, dropna=False)

mixed_zoning_flag
0    0.713333
1    0.286667
Name: proportion, dtype: float64

In [20]:
# lot size distribution
site_features["lot_size_proxy_sqm"].describe()

count    4.995000e+04
mean     2.821956e+05
std      7.710699e+06
min      4.840958e-01
25%      8.462787e+02
50%      1.239546e+03
75%      3.812829e+03
max      9.411130e+08
Name: lot_size_proxy_sqm, dtype: float64

In [21]:
print(site_features["primary_zoning_code"].isna().mean())

0.0026626626626626627


In [22]:
print(site_features["mixed_zoning_flag"].mean())

0.2866666666666667


In [23]:
print(site_features["primary_zoning_code"].value_counts(dropna=False).head(20))

primary_zoning_code
R2     18373
R1      6505
R3      4709
R4      2643
MU1     2196
RU1     2114
SP2     1970
RE1     1651
E1      1255
RU5     1049
C4       970
E4       869
R5       745
RU2      697
E2       637
C2       575
C3       452
RU4      334
SP5      318
E3       313
Name: count, dtype: int64


In [24]:
print(site_features["lot_size_proxy_sqm"].describe())

count    4.995000e+04
mean     2.821956e+05
std      7.710699e+06
min      4.840958e-01
25%      8.462787e+02
50%      1.239546e+03
75%      3.812829e+03
max      9.411130e+08
Name: lot_size_proxy_sqm, dtype: float64


In [25]:
site_features["is_large_lot_outlier"] = (
    site_features["lot_size_proxy_sqm"] > site_features["lot_size_proxy_sqm"].quantile(0.999)
).astype(int)

site_features["is_tiny_lot_outlier"] = (
    site_features["lot_size_proxy_sqm"] < site_features["lot_size_proxy_sqm"].quantile(0.001)
).astype(int)

In [26]:
site_features["is_large_lot_outlier"].value_counts()
site_features["is_tiny_lot_outlier"].value_counts()

is_tiny_lot_outlier
0    49901
1       49
Name: count, dtype: int64

In [27]:
import numpy as np
site_features["log_lot_size_proxy_sqm"] = np.log1p(site_features["lot_size_proxy_sqm"])
site_features["log_lot_size_proxy_sqm"].describe()

count    49950.000000
mean         7.802438
std          1.904749
min          0.394806
25%          6.742030
50%          7.123307
75%          8.246389
max         20.662574
Name: log_lot_size_proxy_sqm, dtype: float64

In [28]:
Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

site_features.to_parquet(
    "../../data/processed/site_features/property_site_features_zoning_sample.parquet",
    index=False,
)